# Exact orientation: when floating-point signs become unreliable

The orientation predicate asks whether three points make a counter-clockwise turn, a clockwise turn or are collinear. This notebook compares the ordinary floating-point determinant with an exact rational reference calculation.

The implementation uses Python Fraction arithmetic. It is a teaching oracle, not an adaptive Shewchuk predicate.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt

root = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (p / 'implementations/python/exact-orientation').is_dir()), None)
if root is None:
    raise RuntimeError('Run this notebook from a repository checkout.')
sys.path.insert(0, str(root / 'implementations/python/exact-orientation'))
from exact_orientation import orientation_sign


## 1. The determinant

For points a, b and c, calculate

$$D=(b_x-a_x)(c_y-a_y)-(b_y-a_y)(c_x-a_x).$$

The sign gives the turn direction. The magnitude is not the classification we need.

In [ ]:
a, b = (0, 0), (4, 0)
examples = {'counter-clockwise': (a, b, (1, 3)),
            'clockwise': (a, b, (1, -3)),
            'collinear': (a, b, (2, 0))}
for label, (p, q, r) in examples.items():
    print(label, '=>', orientation_sign(p, q, r))


## 2. A near-degenerate floating-point example

These coordinates are already binary floating-point values. The ordinary calculation rounds the determinant to zero, but exact rational evaluation of those represented values gives a negative sign. Exact arithmetic cannot recover the intended real-world values before they were rounded; it only evaluates the supplied representation exactly.

In [ ]:
a = (5528636682.011507, -1595419105.704565)
b = (3096585648.789674, 6380875131.612917)
c = (4421959453.453243, 2034103087.7611067)
naive = ((b[0]-a[0])*(c[1]-a[1])
         - (b[1]-a[1])*(c[0]-a[0]))
exact_sign = orientation_sign(a, b, c)
print('Naive floating-point determinant:', naive)
print('Exact sign for represented inputs:', exact_sign)


## 3. Why the sign matters

A predicate controls algorithm branches. A wrong sign can change whether an edge is flipped, whether a segment is considered to cross, or which side of a boundary contains a point. This is a topological error, not just a slightly inaccurate coordinate.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot([0, 4], [0, 0], marker='o', label='Directed baseline a → b')
ax.scatter([1, 1, 2], [2, -2, 0], s=65, label='c: left, right, on-line')
ax.annotate('counter-clockwise', (1, 2), xytext=(1.3, 2.5), arrowprops={'arrowstyle':'->'})
ax.annotate('clockwise', (1, -2), xytext=(1.3, -2.5), arrowprops={'arrowstyle':'->'})
ax.annotate('collinear', (2, 0), xytext=(2.3, 0.5), arrowprops={'arrowstyle':'->'})
ax.set_xlim(-0.5, 5); ax.set_ylim(-3.2, 3.2); ax.set_aspect('equal')
ax.set_title('Orientation is a sign classification')
ax.legend(loc='upper right')
plt.show()


## 4. Production strategies

- Exact rational arithmetic is a useful reference, but may be expensive.
- A proven floating-point filter can return quickly when the sign is safely separated from zero.
- Adaptive precision computes more accurately only when the fast result cannot be certified.
- A fixed epsilon is not a proof of correctness and changes meaning with coordinate scale.

## Investigate

1. Swap two points and confirm that the sign reverses.
2. Translate integer coordinates by a very large exact offset.
3. Generate random integer triples and compare a naive determinant with the exact reference.
4. Explain why exact predicates and exact constructions are different guarantees.

Continue with the [method lesson](../methods/exact-orientation-predicate.md), [implementation walkthrough](../implementations/python/exact-orientation/implementation-walkthrough.md) and [exercise](../exercises/exact-orientation.md).